# 2. Injection des attaques de transmission (MITM et injection)

**Objectif.** Fabriquer le jeu de données d'étude : les traces réelles, dans lesquelles un attaquant
modifie les mesures **pendant leur trajet** entre le capteur et le broker MQTT. Le capteur fonctionne,
mais le serveur reçoit une fausse valeur.

| Scénario | Capteur visé | Ce que fait l'attaquant |
| --- | --- | --- |
| `MITM_gel` | frigo | répète la dernière valeur en boucle |
| `MITM_rejeu` | frigo | renvoie une fenêtre passée (au moins 1 h plus tôt) |
| `Injection_nominale` | frigo | envoie des valeurs forgées autour de 7 °C, peu bruitées |
| `Injection_biais_2.0` | frigo | décale toutes les mesures de −2 °C |
| `Injection_derive` | frigo | décale peu à peu, de 0 à −2 °C |
| `MITM_gel_thermostat` | salle | répète la dernière valeur |
| `Injection_biais_thermostat` | salle | décale de −1,5 °C |

> **Périmètre.** Les biais faibles `Injection_biais_0.5` (−0,5 °C) et `Injection_biais_1.0`
> (−1,0 °C) ont été **retirés du périmètre d'étude** : leur amplitude est **sous le bruit du
> capteur** (σ ≈ 3,4 °C), ils sont quasi indétectables (AUC ≈ 0,64 et 0,82). Ils restent
> documentés comme **limite connue** du système (voir `Soutenance/`).

Le « masquage » (cacher une vraie hausse) est abandonné : ces traces n'ont pas d'excursion réelle à cacher.

In [ ]:
%%capture
import sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend").is_dir())
sys.path[:0] = [str(ROOT / "backend"), str(ROOT / "backend" / "scripts")]
import build_dataset as bd
import transmission_real as tr
import transmission_models as tm
import transmission_study as ts
from app.api.v1 import phase2
from app.core import paths

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (11, 3.5)
SEED = tr.SEED
COLS = {"weather": "temperature", "fridge": "fridge_temperature", "thermostat": "current_temperature"}
EPISODES_PATH = paths.PROCESSED_DIR / "episodes_transmission_reel.json"
MODEL_DIR = paths.MODELS_DIR / "transmission_reel"

In [ ]:
base = pd.read_csv(paths.PROCESSED_DIR / "traces_reelles_5s.csv", parse_dates=["datetime"])
base["fridge_cond"] = base["fridge_cond"].astype(str)
print(f"{len(base):,} lignes de traces réelles fusionnées")

## 2.1 Découpage en blocs et planning des attaques

- **Le découpage train/val/test n'est plus stocké dans le jeu de données.** Il est calculé
  **au moment de l'entraînement** (fonction `transmission_real.block_split`), de façon
  **non aléatoire** : les heures sont découpées en **blocs d'une heure** attribués selon un
  motif fixe de **70 % train / 15 % val / 15 % test** (14 train / 3 val / 3 test sur un cycle
  de 20 blocs). Une attaque tient **entièrement** dans un bloc → **aucune attaque n'est
  coupée** entre deux rôles (pas de fuite temporelle).
- Un **épisode d'attaque de 15 à 25 minutes** est placé dans **chaque bloc** (train, val et
  test). Le jeu de données est donc à **~33 % d'attaques** : les modèles supervisés
  (Random Forest, XGBoost…) ont assez d'exemples d'attaque pour apprendre.
- Après chaque épisode, 5 minutes sont exclues de l'évaluation (les fenêtres glissantes traînent encore l'attaque).
- Les scénarios se suivent en boucle, pour que chacun apparaisse dans chaque rôle.


In [ ]:
rng = np.random.default_rng(SEED)
episodes = tr.build_schedule(len(base), rng, run=0, roles=("train", "val", "test"))
plan = pd.DataFrame(episodes)
plan["début"] = base["datetime"].iloc[plan["start"]].values
plan["durée (min)"] = (plan["end"] - plan["start"]) * 5 / 60
plan[["name", "sensor", "kind", "param", "split", "début", "durée (min)"]]

In [ ]:
df = tr.inject(base, episodes, rng, run=0)
# Le découpage train/val/test n'est PLUS stocké dans le dataset : on le recalcule ici
# (non aléatoire, 70/15/15, par blocs d'une heure) rien que pour lire le bilan.
role = tr.block_split(len(df), run=0)
bilan = df.groupby(role)["label"].agg(lignes="size", attaques="sum")
bilan["part d'attaques"] = (100 * bilan["attaques"] / bilan["lignes"]).round(1).astype(str) + " %"
bilan


## 2.2 À quoi ressemble chaque attaque ?

In [ ]:
def montrer(kind, marge=120):
    ep = next(e for e in episodes if e["kind"] == kind)
    col = "fridge" if ep["sensor"] == "fridge" else "thermostat"
    a, b = ep["start"] - marge, ep["end"] + marge
    t = df["datetime"].iloc[a:b]
    plt.figure(figsize=(11, 3))
    plt.plot(t, df[f"{col}_true"].iloc[a:b], lw=0.8, label="valeur réelle du capteur")
    plt.plot(t, df[f"{col}_temp"].iloc[a:b], lw=0.8, label="valeur reçue par le serveur")
    plt.axvspan(df["datetime"].iloc[ep["start"]], df["datetime"].iloc[ep["end"] - 1], color="red", alpha=0.08, label="attaque")
    plt.title(f"{ep['name']} ({ep['split']})"); plt.ylabel(f"{col} (°C)"); plt.legend(loc="upper right"); plt.tight_layout()

for kind in ("gel", "rejeu", "nominale", "biais", "derive"):
    montrer(kind)

Le gel et l'injection nominale se voient à l'œil : la mesure ne bouge plus, ou devient trop
régulière. Le rejeu et les biais sont invisibles à l'œil : les valeurs rejouées ou décalées ont
la même allure que les vraies. Seul un calcul (niveau moyen, séquence déjà vue) peut les repérer.

## 2.3 Features et enregistrement du jeu de données

In [ ]:
df, feats = tm.engineer(df)
df.to_csv(tr.DATASET_PATH, index=False, float_format="%.4f")
with open(EPISODES_PATH, "w", encoding="utf-8") as f:
    json.dump(episodes, f, indent=1)
print("Jeu de données :", tr.DATASET_PATH, f"({len(df):,} lignes, {len(feats)} features)")
print("Planning des attaques :", EPISODES_PATH, f"({len(episodes)} épisodes)")

Ce fichier correspond à la **répétition 0** de l'étude. Comme les traces ne durent que 62 heures,
le script `backend/scripts/transmission_models.py` rejoue 9 fois ce découpage en faisant tourner les
rôles des heures et les scénarios, puis cumule les résultats. Les notebooks suivants travaillent sur
la répétition 0 et affichent aussi le résultat cumulé.